# QuantJourney SDK - Earnings Event PEAD Research

This notebook demonstrates a QuantJourney SDK workflow that joins earnings calendar, earnings surprises, fundamentals and prices to study post-earnings announcement drift across a peer set.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'AMZN', 'META']
calendar = qj.fmp.get_earnings_calendar(from_date='2023-01-01', to_date=END)
surprises = {symbol: qj.fmp.get_earnings_surprises(symbol=symbol) for symbol in symbols}
ratios = {symbol: qj.fmp.get_financial_ratios_ttm(symbol=symbol) for symbol in symbols}
prices, volumes = price_panel(symbols, start='2022-01-01', end=END)


In [ ]:
event_rows = []
for symbol, payload in surprises.items():
    for item in as_rows(payload):
        event_date = pd.to_datetime(item.get('date') or item.get('fiscalDateEnding'), errors='coerce')
        surprise = pd.to_numeric(item.get('surprisePercentage') or item.get('surprise'), errors='coerce')
        if pd.notna(event_date):
            event_rows.append({'symbol': symbol, 'event_date': event_date, 'surprise': surprise})
events = pd.DataFrame(event_rows)
if events.empty:
    events = pd.DataFrame({'symbol': symbols, 'event_date': [prices.index[-90]] * len(symbols), 'surprise': np.nan})


In [ ]:
curves = []
event_metrics = []
for row in events.dropna(subset=['event_date']).itertuples():
    if row.symbol not in prices:
        continue
    idx = prices.index.searchsorted(row.event_date)
    if idx < 10 or idx + 42 >= len(prices):
        continue
    window = prices[row.symbol].iloc[idx - 10:idx + 43]
    curve = window / window.iloc[10] - 1
    curves.append(pd.Series(curve.values, index=range(-10, len(curve) - 10), name=row.symbol))
    event_metrics.append({'symbol': row.symbol, 'event_date': row.event_date, 'surprise': row.surprise, 'fwd_5d': curve.iloc[15], 'fwd_21d': curve.iloc[31], 'fwd_42d': curve.iloc[-1]})
event_curve = pd.concat(curves, axis=1) if curves else pd.DataFrame()
metrics = pd.DataFrame(event_metrics)
display(metrics.sort_values('fwd_21d', ascending=False).head(20))
if not event_curve.empty:
    event_curve.mean(axis=1).plot(title='Average PEAD curve')
    plt.axvline(0, color='black', linestyle='--', alpha=0.5)
    plt.ylabel('return vs event day')
    plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.